# Arabic Moonshine LoRA Training on Google Colab

This notebook runs the Arabic LoRA fine-tuning workflow for `UsefulSensors/moonshine-base-ar`.

Use a GPU runtime in Colab: **Runtime > Change runtime type > GPU**. The notebook assumes your dataset zip or extracted dataset is stored in Google Drive.


## 1. Runtime Check

Confirm that Colab attached a GPU. CPU will work mechanically, but it will be slow.

In [ ]:
!nvidia-smi || true


## 2. Configure Paths

Edit these variables if your GitHub repo, branch, or Drive dataset path differs.

In [ ]:
REPO_URL = "https://github.com/MarshalXu/finetune-moonshine-asr.git"
BRANCH = "shawnx/peft_ft"
WORKDIR = "/content/finetune-moonshine-asr"

DRIVE_ROOT = "/content/drive/MyDrive/moonshine"
DATA_ZIP = f"{DRIVE_ROOT}/whisper_ar_manifist0508.zip"

DATASET_NAME = "whisper_ar_manifist0508"
RAW_DATA_DIR = f"{WORKDIR}/datasets/{DATASET_NAME}"
HF_DATASET_DIR = f"{WORKDIR}/datasets/{DATASET_NAME}_hf"
OUTPUT_DIR = f"{WORKDIR}/results-moonshine-base-ar-lora-colab"
DRIVE_OUTPUT_DIR = f"{DRIVE_ROOT}/results-moonshine-base-ar-lora-colab"

print("Repo:", REPO_URL)
print("Branch:", BRANCH)
print("Dataset zip:", DATA_ZIP)
print("Output:", OUTPUT_DIR)


## 3. Clone the Project

This pulls the branch that contains the Arabic LoRA workflow and Colab notebook.

In [ ]:
import os
import subprocess


def run(cmd):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=True)

run(f"rm -rf {WORKDIR}")
run(f"git clone --branch {BRANCH} {REPO_URL} {WORKDIR}")
os.chdir(WORKDIR)
print("cwd:", os.getcwd())


## 4. Install Dependencies

Colab already provides PyTorch. This installs the project dependencies, including `peft` and `torchcodec`.

In [ ]:
!python -m pip install -q -r requirements.txt
!python - <<'PYCODE'
import torch
import transformers
import datasets
import peft
import torchcodec
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
print("transformers", transformers.__version__)
print("datasets", datasets.__version__)
print("peft", peft.__version__)
print("torchcodec", torchcodec.__version__)
PYCODE


## 5. Mount Google Drive

Put `whisper_ar_manifist0508.zip` under `DRIVE_ROOT`, or edit `DATA_ZIP` above.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
!mkdir -p "$DRIVE_ROOT"


## 6. Prepare Raw Dataset

This unzips your dataset from Drive into the repo `datasets/` directory. If the raw dataset is already extracted there, this cell does nothing.

In [ ]:
from pathlib import Path

raw_dir = Path(RAW_DATA_DIR)
zip_path = Path(DATA_ZIP)

if raw_dir.exists():
    print(f"Raw dataset already exists: {raw_dir}")
elif zip_path.exists():
    print(f"Unzipping {zip_path} -> {raw_dir.parent}")
    raw_dir.parent.mkdir(parents=True, exist_ok=True)
    run(f"unzip -q -o {zip_path} -d {raw_dir.parent}")
    if not raw_dir.exists():
        raise FileNotFoundError(f"Expected extracted dataset at {raw_dir}. Check zip top-level folder name.")
else:
    raise FileNotFoundError(
        f"Neither raw dataset nor zip exists. Expected raw dir {raw_dir} or zip {zip_path}"
    )

print("train manifest:", raw_dir / "train.jsonl", (raw_dir / "train.jsonl").exists())
print("test manifest:", raw_dir / "test.jsonl", (raw_dir / "test.jsonl").exists())


## 7. Convert and Validate Dataset

This creates a Hugging Face `DatasetDict` and skips audio files that `torchcodec` cannot decode. This can take several minutes because it validates every audio file.

In [ ]:
!python scripts/prepare_manifest_dataset.py \
  --manifest-dir "$RAW_DATA_DIR" \
  --output "$HF_DATASET_DIR" \
  --validate-audio \
  --skip-invalid-audio \
  --overwrite


## 8. Create a Colab Training Config

This copies the repo LoRA config and overrides output paths for Colab. The default is 6 epochs, evaluation and checkpoint saving every epoch, and best model selected by WER.

In [ ]:
import yaml
from pathlib import Path

base_config_path = Path("configs/moonshine_base_ar_lora_cpu_train.yaml")
colab_config_path = Path("configs/moonshine_base_ar_lora_colab.yaml")
config = yaml.safe_load(base_config_path.read_text())
config["dataset"]["path"] = HF_DATASET_DIR
config["training"]["output_dir"] = OUTPUT_DIR
config["training"]["logging_dir"] = f"{WORKDIR}/logs/moonshine-base-ar-lora-colab"

# GPU-friendly baseline. Adjust if you hit memory limits.
config["training"]["per_device_train_batch_size"] = 4
config["training"]["per_device_eval_batch_size"] = 4
config["training"]["gradient_accumulation_steps"] = 4
config["training"]["fp16"] = True
config["training"]["fp16_full_eval"] = True

colab_config_path.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False))
print(colab_config_path.read_text())


## 9. Start Training

Logs are shown live. `eval_wer` and `eval_cer` appear at the end of each epoch. The best adapter is loaded at the end according to WER.

In [ ]:
!python -u train.py --config configs/moonshine_base_ar_lora_colab.yaml 2>&1 | tee "$DRIVE_ROOT/lora_train.log"


## 10. Resume Training

Use this if Colab disconnects. Change the checkpoint path to the latest checkpoint in your output directory.

In [ ]:
!find "$OUTPUT_DIR" -maxdepth 1 -type d -name 'checkpoint-*' | sort

# Example:
# !python -u train.py --config configs/moonshine_base_ar_lora_colab.yaml \
#   --resume "$OUTPUT_DIR/checkpoint-XXXX" 2>&1 | tee -a "$DRIVE_ROOT/lora_train_resume.log"


## 11. Copy Artifacts to Drive

The final output is an adapter, not a full model. For inference later, load the base model plus this adapter.

In [ ]:
!mkdir -p "$DRIVE_OUTPUT_DIR"
!rsync -av "$OUTPUT_DIR/" "$DRIVE_OUTPUT_DIR/"
!find "$DRIVE_OUTPUT_DIR" -maxdepth 2 -type f | sort | sed -n '1,80p'
